---
## 🏁 CÉLULA 17 — Exercício final

Coloque em prática tudo que você aprendeu hoje.  
Resolva os passos abaixo **nesta célula** (pode criar mais células se quiser).

---

### 🏭 Cenário
O gestor de RH da Indústria Metalúrgica Catarinense pediu 2 entregáveis:
1. A base de funcionários em Excel com **uma aba por departamento**
2. Um relatório de tempo de casa pronto para o Power BI

---

### Passos

**1.** Carregue o `base_rh.csv` com os parâmetros corretos e confirme que `Salario` é `float64`.

**2.** Use `df.info()` e `df.describe()` para fazer o diagnóstico. Anote em uma célula Markdown:  
→ *"Os 3 problemas mais importantes que encontrei foram..."*

**3.** Converta `Data_Admissao` para datetime. Crie as colunas:  
`Ano_Admissao`, `Mes_Admissao`, `Tempo_Casa_Anos` e `Faixa_Tempo_Casa`.

**4.** Responda com `groupby`:  
→ Qual departamento tem a **maior** média de tempo de casa?  
→ Qual tem a **menor**?

**5.** Exporte `base_rh_deptos.xlsx` com uma aba `'Completo'` e mais uma aba por departamento.

**6.** Faça o commit:  
```bash
git pull origin master
git add alunos/seunome/
git commit -m "semana 04 - dia 01: entrega arquivos e datetime"
git push origin master
git log --oneline -3
```
Cole o resultado do `git log` na última célula do notebook.

---

> 💡 **Dica para o Passo 4:**  
> `df.groupby('Departamento')['Tempo_Casa_Anos'].mean().sort_values(ascending=False)`

> ⚠️ **Se o push falhar com 'rejected':** você esqueceu o `git pull` antes.

In [ ]:
# ── Resolva o exercício aqui ──────────────────────────────────────────────
# Passo 1: carregue o CSV
import pandas as pd
import re

print("\nVersão do pandas:", pd.__version__)
print("\nBibliotecas importadas com sucesso!")

# Caminho do CSV
URL = (
    "http://raw.githubusercontent.com/cfneves/"
    "turma-visualizacao-de-dados/master/alunos/"
    "luiz_fernando_jesus/semana_04/base_rh.csv"
)

# Lendo o CSV
df = pd.read_csv(
    URL,
    sep=';',
    encoding='cp1252',
    decimal=','
)

# Total de linhas e colunas
print("Arquivo lido com sucesso!")
print(f"\nTotal de linhas: {df.shape[0]} X Total de colunas: {df.shape[1]}")

# Primeiras linhas do df
print("\n========= PRIMEIRAS LINHAS DO DF =========")
df.head(3)


Versão do pandas: 3.0.2

Bibliotecas importadas com sucesso!
Arquivo lido com sucesso!

Total de linhas: 1000 X Total de colunas: 10

========= PRIMEIRAS LINHAS DO DF =========


,ID_Funcionario,Nome,Departamento,Cargo,Salario,Data_Admissao,Genero,Idade,Estado_Civil,Status
0,1,Julia Nunes,Logística,Coordenador,9088.34,13/08/2024,F,43,Solteiro,Inativo
1,2,Sr. Gustavo Duarte,TI,Gerente,8155.98,29/04/2017,F,59,Solteiro,Inativo
2,3,Srta. Mariana Cunha,RH,Coordenador,14027.93,11/12/2024,F,27,Divorciado,Ativo


In [3]:
# Passo 2: diagnóstico
# Diagnóstico do dataframe:
print("============ INFORMAÇÕES DO DF: ============")
print(f"{df.info()}")

print("\n============ DESCRIÇÃO DO DF: ============")
print(f"{df.describe().round(2)}")

============ INFORMAÇÕES DO DF: ============
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID_Funcionario  1000 non-null   int64  
 1   Nome            1000 non-null   str    
 2   Departamento    1000 non-null   str    
 3   Cargo           1000 non-null   str    
 4   Salario         1000 non-null   float64
 5   Data_Admissao   1000 non-null   str    
 6   Genero          1000 non-null   str    
 7   Idade           1000 non-null   int64  
 8   Estado_Civil    1000 non-null   str    
 9   Status          1000 non-null   str    
dtypes: float64(1), int64(2), str(7)
memory usage: 78.3 KB
None

============ DESCRIÇÃO DO DF: ============
       ID_Funcionario   Salario    Idade
count         1000.00   1000.00  1000.00
mean           500.50   8579.95    41.40
std            288.82   3657.37    13.67
min              1.00   2000.71    18.00
25%    

# Passo 3: converter datas e criar colunas derivadas
# PROBLEMAS NO DF

1º Data está no formato de str: deve ser corrigido para (datetime)

2º Colunas que estão faltando para uma analise mais detalhada.

3º ...

In [4]:
# 1º resolvendo o primeiro problema:
# Conveter uma coluna inteira para data:

# Criando uma cópia do df

df_limpo = df.copy()

df_limpo["Data_Admissao"] = pd.to_datetime(
    df_limpo["Data_Admissao"], format="%d/%m/%Y", errors="coerce"
)
# Coluna Data_Admissao convertida.

In [5]:
# 2. Criando novas colunas derivadas da Data_Admissao:

df_limpo["Ano_Admissao"] = df_limpo["Data_Admissao"].dt.year
df_limpo["Mes_Admissao"] = df_limpo["Data_Admissao"].dt.month

hoje_ts = pd.Timestamp.today()

df_limpo["Tempo_Casa_Anos"] = (
    (hoje_ts - df_limpo["Data_Admissao"]).dt.days / 365.25
).round(1)

In [6]:
# Função que classifica o tempo de casa que o usuario tem.
def classificar_tempo(anos):
    if pd.isna(anos):
        return "Sem data"
    elif anos < 1:
        return "Menos de 1 ano"
    elif anos < 3:
        return "1 a 3 anos"
    elif anos < 5:
        return "3 a 5 anos"
    else:
        return "Mais de 5 anos"


df_limpo["Faixa_Tempo_Casa"] = df_limpo["Tempo_Casa_Anos"].apply(classificar_tempo)

In [7]:
# Passo 4: groupby — maior e menor média de tempo de casa
print("========= ANALISE DE TEMPO DE CASA POR DEPARTAMENTO =========")
print(
    df_limpo.groupby("Departamento")["Tempo_Casa_Anos"]
    .mean()
    .round(2)
    .sort_values(ascending=False)
)


========= ANALISE DE TEMPO DE CASA POR DEPARTAMENTO =========
Departamento
RH            5.82
Vendas        5.67
TI            5.54
Produção      5.52
Logística     5.34
Financeiro    5.15
Name: Tempo_Casa_Anos, dtype: float64


In [8]:
# Passo 5: exportar para Excel
print('========= EXPORTANDO PARA EXCEL =========')

with pd.ExcelWriter('base_rh_deptos.xlsx', engine='openpyxl') as writer:
    df_limpo.to_excel(writer, sheet_name='Todos', index=False)
    print('Aba Todos: Todos os registros')

    for depto in sorted(df['Departamento'].unique()):
        df_depto = df[df['Departamento'] == depto]
        nome_aba = depto[:31]
        df_depto.to_excel(writer, sheet_name=nome_aba, index=False)
        print(f'Aba"{nome_aba}": {len(df_depto)} registros')
print('\nbase_rh_depto.xlsx gerado!')

# Passo 6: cole o output do git log aqui como comentário
# git log:

========= EXPORTANDO PARA EXCEL =========
Aba Todos: Todos os registros
Aba"Financeiro": 189 registros
Aba"Logística": 156 registros
Aba"Produção": 182 registros
Aba"RH": 166 registros
Aba"TI": 147 registros
Aba"Vendas": 160 registros

base_rh_depto.xlsx gerado!
